# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sayuj5/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import subprocess, os
from google.colab import userdata

if not os.path.exists('/content/flyrank-internship-ml'):
    subprocess.run(['git', 'clone', 'https://github.com/sayuj5/flyrank-internship-ml.git'],
                   capture_output=True, text=True)
    print("Repo cloned")
else:
    print("Repo already exists")

import huggingface_hub
token = userdata.get('HF_TOKEN')
huggingface_hub.login(token=token, add_to_git_credential=False)
print("HF login done")

Repo already exists
HF login done


In [9]:
import pandas as pd

parquet_path = huggingface_hub.hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=token
)

df = pd.read_parquet(parquet_path)
df['report_date'] = pd.to_datetime(df['report_date'])
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Shape: (9841378, 30)
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [10]:
import numpy as np
import os

# Filter to GSC-available rows only
df_avail = df[df['gsc_data_available'] == True].copy()

# Aggregate to one row per article over March 2026
art = df_avail.groupby(['client_hash_id', 'content_hash_id']).agg(
    impressions    = ('gsc_impressions', 'sum'),
    clicks         = ('gsc_clicks', 'sum'),
    avg_position   = ('gsc_avg_position', 'mean'),
    engaged_sessions = ('ga4_engaged_sessions', 'sum'),
    scroll_events  = ('scroll_events', 'sum'),
    days_active    = ('report_date', 'nunique'),
    pageviews      = ('ga4_pageviews', 'sum'),
).reset_index()

# Derived signals
art['ctr'] = art['clicks'] / art['impressions'].replace(0, np.nan)
art['clicks_per_day'] = art['clicks'] / art['days_active']

print(f"Articles in March 2026 (GSC available): {len(art):,}")
print(f"Columns: {art.columns.tolist()}")
art.head(5)

Articles in March 2026 (GSC available): 176,738
Columns: ['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'avg_position', 'engaged_sessions', 'scroll_events', 'days_active', 'pageviews', 'ctr', 'clicks_per_day']


,client_hash_id,content_hash_id,impressions,clicks,avg_position,engaged_sessions,scroll_events,days_active,pageviews,ctr,clicks_per_day
0,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.000000,0.0,0.0,1,0.0,0.000000,0.000000
1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331,2,14.129210,0.0,0.0,31,0.0,0.006042,0.064516
2,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,0,9.225529,0.0,0.0,6,0.0,0.000000,0.000000
3,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,145,0,8.470926,0.0,0.0,30,0.0,0.000000,0.000000
4,client_0797ff3a1fc9a6a5,content_1207efddce873942,461,0,14.859827,0.0,0.0,31,0.0,0.000000,0.000000


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: Low-CTR High-Impression Content Refresh Scorer

Signal check 1 — CTR vs Average Position (flag-linked: CTR-fix logic)
Hypothesis: articles ranking in positions 1-10 but with CTR below 1% are
underperforming relative to their visibility. This is the signal behind
FlyRank's CTR-fix flag.

Signal check 2 — Impressions vs Clicks (flag-linked: quick-win logic)
Hypothesis: articles with high impressions (>=100) but zero or near-zero
clicks signal a content/relevance mismatch worth fixing.

Rule encoding:
- Score = impressions × (0.01 - ctr)  [only where ctr < 0.01 and impressions >= 100]
- Reason code: LOW_CTR_HIGH_IMP
- Action label: REFRESH_CONTENT

In [11]:
import numpy as np

# Signal Check 1: CTR by position bucket
df_s1 = art[art['ctr'].notna() & art['avg_position'].notna()].copy()
bins = [0, 3, 10, 20, 50, 100, float('inf')]
labels = ['1-3', '4-10', '11-20', '21-50', '51-100', '100+']
df_s1['position_bucket'] = pd.cut(df_s1['avg_position'], bins=bins, labels=labels)

s1 = df_s1.groupby('position_bucket', observed=True).agg(
    n=('ctr','count'),
    mean_ctr=('ctr','mean'),
    median_ctr=('ctr','median'),
).reset_index()

print("=== Signal 1: CTR by Position Bucket ===")
print(s1.to_string(index=False))
print(f"n total: {len(df_s1):,}")
print("Verdict: CONFIRMED — positions 1-10 have higher CTR than 20+")

print()

# Signal Check 2: Clicks by impressions bucket
bins2 = [0, 10, 50, 200, 1000, 5000, float('inf')]
labels2 = ['1-10','11-50','51-200','201-1k','1k-5k','5k+']
art['imp_bucket'] = pd.cut(art['impressions'], bins=bins2, labels=labels2)

s2 = art.groupby('imp_bucket', observed=True).agg(
    n=('clicks','count'),
    mean_clicks=('clicks','mean'),
    zero_click_pct=('clicks', lambda x: (x==0).mean()*100)
).reset_index()

print("=== Signal 2: Clicks by Impressions Bucket ===")
print(s2.to_string(index=False))
print(f"n total: {len(art):,}")
print("Verdict: CONFIRMED — many high-impression articles have zero clicks (quick-win targets)")

=== Signal 1: CTR by Position Bucket ===
position_bucket     n  mean_ctr  median_ctr
            1-3 16144  0.010589         0.0
           4-10 81988  0.004926         0.0
          11-20 32203  0.003211         0.0
          21-50 33288  0.002287         0.0
         51-100 11579  0.000857         0.0
           100+   102  0.006127         0.0
n total: 176,738
Verdict: CONFIRMED — positions 1-10 have higher CTR than 20+

=== Signal 2: Clicks by Impressions Bucket ===
imp_bucket     n  mean_clicks  zero_click_pct
      1-10 34910     0.033658       96.917789
     11-50 26105     0.097874       92.101130
    51-200 31014     0.268072       81.295544
    201-1k 39674     1.197358       51.502243
     1k-5k 31745     7.287919       13.038274
       5k+ 13290    39.949511        1.798345
n total: 176,738
Verdict: CONFIRMED — many high-impression articles have zero clicks (quick-win targets)


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Encoding the rule: score each article by how much CTR opportunity it wastes.
Score = impressions × (0.01 - ctr) where impressions >= 100 and ctr < 0.01
Higher score = more impressions wasted at low CTR = higher priority refresh.
Reason code: LOW_CTR_HIGH_IMP | Action: REFRESH_CONTENT

In [12]:
import os

scored = art[
    (art['impressions'] >= 100) &
    (art['ctr'].notna()) &
    (art['ctr'] <= 0.01)
].copy()

scored['score'] = scored['impressions'] * (0.01 - scored['ctr'])
scored['reason_code'] = 'LOW_CTR_HIGH_IMP'
scored['action_label'] = 'REFRESH_CONTENT'
scored = scored.sort_values('score', ascending=False).reset_index(drop=True)
scored['rank'] = scored.index + 1

print(f"Articles flagged: {len(scored):,}")
print(f"Score range: {scored['score'].min():.2f} to {scored['score'].max():.2f}")
print(f"\nTop 10:")
print(scored[['rank','content_hash_id','impressions','ctr','score',
              'reason_code','action_label']].head(10).to_string(index=False))

# Write CSV
os.makedirs('/content/flyrank-internship-ml/work/outputs', exist_ok=True)
out_path = '/content/flyrank-internship-ml/work/outputs/baseline_action_score.csv'
scored[['client_hash_id','content_hash_id','impressions','clicks',
        'avg_position','ctr','score','reason_code','action_label','rank']].to_csv(out_path, index=False)
print(f"\n✓ CSV written: {out_path} ({len(scored):,} rows)")

Articles flagged: 96,862
Score range: 0.00 to 2100.04

Top 10:
 rank          content_hash_id  impressions      ctr   score      reason_code    action_label
    1 content_44f34c0a90047651       212404 0.000113 2100.04 LOW_CTR_HIGH_IMP REFRESH_CONTENT
    2 content_e8a52cf3d5988c07       244931 0.002731 1780.31 LOW_CTR_HIGH_IMP REFRESH_CONTENT
    3 content_8d7d99f109e19aa2       203497 0.001420 1745.97 LOW_CTR_HIGH_IMP REFRESH_CONTENT
    4 content_36e53e9c707674fc       194579 0.001244 1703.79 LOW_CTR_HIGH_IMP REFRESH_CONTENT
    5 content_b99ea6861864dea5       194337 0.001858 1582.37 LOW_CTR_HIGH_IMP REFRESH_CONTENT
    6 content_0e03de7680314cd5       221310 0.003253 1493.10 LOW_CTR_HIGH_IMP REFRESH_CONTENT
    7 content_acbcc847f8996314       170808 0.001534 1446.08 LOW_CTR_HIGH_IMP REFRESH_CONTENT
    8 content_34a70fea29d15f24       143019 0.000301 1387.19 LOW_CTR_HIGH_IMP REFRESH_CONTENT
    9 content_82e35c4845e6c391       143907 0.000417 1379.07 LOW_CTR_HIGH_IMP REFRESH_CONTE

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review: for each article — action, why it's there, what would make it wrong.

In [13]:
top20 = scored.head(20).copy()

print("=== Top-20 Baseline Queue Review ===\n")
for _, row in top20.iterrows():
    print(f"Rank {int(row['rank'])}: {row['content_hash_id']}")
    print(f"  Action: {row['action_label']} | Score: {row['score']:.1f}")
    print(f"  Why: {int(row['impressions'])} impressions, CTR={row['ctr']:.4f} — high visibility, very low click rate")
    print(f"  Wrong if: avg_position > 20 (low CTR expected); navigational query "
          f"(CTR norms differ); article < 30 days old (too early to judge); "
          f"impressions from irrelevant queries.")
    print()

=== Top-20 Baseline Queue Review ===

Rank 1: content_44f34c0a90047651
  Action: REFRESH_CONTENT | Score: 2100.0
  Why: 212404 impressions, CTR=0.0001 — high visibility, very low click rate
  Wrong if: avg_position > 20 (low CTR expected); navigational query (CTR norms differ); article < 30 days old (too early to judge); impressions from irrelevant queries.

Rank 2: content_e8a52cf3d5988c07
  Action: REFRESH_CONTENT | Score: 1780.3
  Why: 244931 impressions, CTR=0.0027 — high visibility, very low click rate
  Wrong if: avg_position > 20 (low CTR expected); navigational query (CTR norms differ); article < 30 days old (too early to judge); impressions from irrelevant queries.

Rank 3: content_8d7d99f109e19aa2
  Action: REFRESH_CONTENT | Score: 1746.0
  Why: 203497 impressions, CTR=0.0014 — high visibility, very low click rate
  Wrong if: avg_position > 20 (low CTR expected); navigational query (CTR norms differ); article < 30 days old (too early to judge); impressions from irrelevant que

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: articles where the rule fires but the flag is likely wrong.

1. Position-blind: rule ignores avg_position. Articles at position 30+
   naturally have low CTR — flagging them for content refresh is wrong;
   they need SEO/ranking work instead.

2. No freshness filter: new articles (<30 days) haven't accumulated clicks
   yet. The rule would wrongly flag them as underperformers.

3. Query-blind: high impressions may come from many unrelated queries.
   The content isn't broken — the keyword targeting is.

Leakage check:
- Score uses only impressions and ctr (both observable before any action)
- No future-window data used (ga4_pageviews excluded — leaky in W03)
- No label-derived columns used
- All inputs are knowable at the decision moment ✓

In [14]:
import json

# Quantify position-blind weakness
print("=== Weak Picks: Position-blind analysis ===")
weak = scored[scored['avg_position'] > 20]
strong = scored[scored['avg_position'] <= 20]
print(f"Flagged with avg_position > 20 (likely noise): {len(weak):,} ({len(weak)/len(scored)*100:.1f}%)")
print(f"Flagged with avg_position <= 20 (genuine):     {len(strong):,} ({len(strong)/len(scored)*100:.1f}%)")
print(f"\nConclusion: {len(weak)/len(scored)*100:.1f}% of flags may be position-driven, not content problems.")
print("Week-5 model must include avg_position as a feature to fix this.")

print("\n=== Leakage Check ===")
print("Features used in score: impressions, ctr")
print("Both are observable BEFORE any editorial action — no leakage ✓")
print("ga4_pageviews excluded (leaks the label — shown in W03) ✓")
print("No future-window data used ✓")

# Save metrics JSON
metrics = {
    "total_flagged": len(scored),
    "position_noise_pct": round(len(weak)/len(scored)*100, 1),
    "genuine_flags_pct": round(len(strong)/len(scored)*100, 1),
    "score_max": round(float(scored['score'].max()), 2),
    "mean_impressions_flagged": round(float(scored['impressions'].mean()), 1),
    "mean_ctr_flagged": round(float(scored['ctr'].mean()), 4),
}
with open('/content/flyrank-internship-ml/work/outputs/w04_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"\nMetrics saved:")
print(json.dumps(metrics, indent=2))

=== Weak Picks: Position-blind analysis ===
Flagged with avg_position > 20 (likely noise): 23,727 (24.5%)
Flagged with avg_position <= 20 (genuine):     73,135 (75.5%)

Conclusion: 24.5% of flags may be position-driven, not content problems.
Week-5 model must include avg_position as a feature to fix this.

=== Leakage Check ===
Features used in score: impressions, ctr
Both are observable BEFORE any editorial action — no leakage ✓
ga4_pageviews excluded (leaks the label — shown in W03) ✓
No future-window data used ✓

Metrics saved:
{
  "total_flagged": 96862,
  "position_noise_pct": 24.5,
  "genuine_flags_pct": 75.5,
  "score_max": 2100.04,
  "mean_impressions_flagged": 2773.1,
  "mean_ctr_flagged": 0.002
}


In [15]:
import subprocess, os
os.chdir('/content/flyrank-internship-ml')
subprocess.run(['git', 'config', 'user.email', 'sayujsur05@gmail.com'], capture_output=True)
subprocess.run(['git', 'config', 'user.name', 'sayuj5'], capture_output=True)
subprocess.run(['git', 'add', 'work/outputs/w04_metrics.json'], capture_output=True)
result = subprocess.run(['git', 'commit', '-m', 'Add W04 baseline metrics'], capture_output=True, text=True)
print(result.stdout or result.stderr)

[main 44bf972] Add W04 baseline metrics
 1 file changed, 8 insertions(+)
 create mode 100644 work/outputs/w04_metrics.json



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.